# Comparación de modelos

## Objetivo

- Conocer a través de métricas cómo se comportan las features v01 y v02.
- Métricas: accuracy, confusion matrix, classification report, feature importance

### Imports y carga de datos

In [84]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [85]:
MATCH_VECTOR_V01_PATH = "../data/processed/match_vector_v01.csv"
MATCH_VECTOR_V02_PATH = "../data/processed/match_vector_v02.csv"

df_v01 = pd.read_csv(MATCH_VECTOR_V01_PATH)
df_v02 = pd.read_csv(MATCH_VECTOR_V02_PATH)

print(df_v01.shape)
print(df_v02.shape)

(128, 60)
(128, 135)


In [86]:
df_v02.columns.tolist()

['match_id',
 'match_date',
 'kick_off',
 'year',
 'home_team',
 'away_team',
 'home_score',
 'away_score',
 'competition_stage',
 'home_Ball Receipt*',
 'home_Ball Recovery',
 'home_Block',
 'home_Carry',
 'home_Clearance',
 'home_Dispossessed',
 'home_Dribble',
 'home_Dribbled Past',
 'home_Duel',
 'home_Error',
 'home_Foul Committed',
 'home_Foul Won',
 'home_Goal Keeper',
 'home_Interception',
 'home_Miscontrol',
 'home_Own Goal Against',
 'home_Pass',
 'home_Player Off',
 'home_Player On',
 'home_Pressure',
 'home_Shot',
 'home_Offside',
 'home_Shield',
 'home_Bad Behaviour',
 'home_50/50',
 'away_Ball Receipt*',
 'away_Ball Recovery',
 'away_Block',
 'away_Carry',
 'away_Clearance',
 'away_Dispossessed',
 'away_Dribble',
 'away_Dribbled Past',
 'away_Duel',
 'away_Error',
 'away_Foul Committed',
 'away_Foul Won',
 'away_Goal Keeper',
 'away_Interception',
 'away_Miscontrol',
 'away_Own Goal Against',
 'away_Pass',
 'away_Player Off',
 'away_Player On',
 'away_Pressure',
 'away_Sh

### Creación de Funciones

#### Función train-test

In [87]:
def split_train_test_by_year(df, target_col="target"):
    train_df = df[df["year"] == 2018].copy()
    test_df = df[df["year"] == 2022].copy()

    columns_to_drop = [
        "match_id",
        "match_date",
        "kick_off",
        "year",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "competition_stage",
        target_col
    ]

    X_train = train_df.drop(columns=columns_to_drop)
    y_train = train_df[target_col]

    X_test = test_df.drop(columns=columns_to_drop)
    y_test = test_df[target_col]

    return X_train, X_test, y_train, y_test

In [88]:
X_train_v01, X_test_v01, y_train, y_test = split_train_test_by_year(df_v01)
X_train_v02, X_test_v02, _, _ = split_train_test_by_year(df_v02)

print("X_train_v01:", X_train_v01.shape)
print("X_test_v01:", X_test_v01.shape)

print("X_train_v02:", X_train_v02.shape)
print("X_test_v02:", X_test_v02.shape)

X_train_v01: (64, 50)
X_test_v01: (64, 50)
X_train_v02: (64, 125)
X_test_v02: (64, 125)


#### Función Para evaluar modelos

In [89]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name=None, dataset_name=None):
    """
    Entrena y evalúa un modelo de predicción.
    Devuelve dict con resultados.
    """
    
    if model_name is None:                       
        model_name = model.__class__.__name__


    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    cm = confusion_matrix(y_test, predictions)

    report = classification_report(
        y_test,
        predictions,
        output_dict=True,
        zero_division = 0
    )

    return {
        "dataset": dataset_name,
        "model_name": model_name,
        "model": model,
        "predictions": predictions,
        "accuracy": accuracy,
        "confusion_matrix": cm,
        "classification_report": report,
    }



#### Funciones para Resultados
(Summarize results, display results, feature importances)

In [90]:
def summarize_results(results):
    """
    Crea un summary DataFrame de una lista de resultados de evaluación de modelos.
    """

    rows = []

    for result in results:
        rows.append({
            "dataset": result["dataset"],
            "model_name": result["model_name"],
            "accuracy": result["accuracy"]
        })

    summary_df = pd.DataFrame(rows)

    summary_df = summary_df.sort_values(
        by="accuracy",
        ascending=False
    ).reset_index(drop=True)

    return summary_df



In [91]:
def display_results(results, labels=[-1, 0, 1]):
    """
    Displays detailed evaluation metrics for each model result.
    """

    for result in results:

        print("=" * 60)
        print(f"{result['model_name']} - {result['dataset']}")
        print("=" * 60)

        print(f"Accuracy: {result['accuracy']:.3f}")

        print("\nConfusion Matrix:")
        cm_df = pd.DataFrame(
            result["confusion_matrix"],
            index=[f"Real {label}" for label in labels],
            columns=[f"Pred {label}" for label in labels]
        )
        display(cm_df)

        print("\nClassification Report:")
        report_df = pd.DataFrame(result["classification_report"]).T
        display(report_df)

        print("\n")
        

In [92]:
def get_feature_importance(result, feature_names, top_n=20):
    """
    Creates a feature importance DataFrame for models with feature_importances_.
    """

    model = result["model"]

    if not hasattr(model, "feature_importances_"):
        raise ValueError(
            f"{result['model_name']} does not have feature_importances_."
        )

    if len(model.feature_importances_) != len(feature_names):
        raise ValueError(
            f"Length mismatch: model has {len(model.feature_importances_)} importances "
            f"but {len(feature_names)} feature names were provided."
        )

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_
    })

    importance_df = importance_df.sort_values(
        by="importance",
        ascending=False
    ).reset_index(drop=True)

    return importance_df.head(top_n)



#### Función para comparar modelos

In [93]:
def run_model_comparison(models, datasets, y_train, y_test):
    """
    Corre varios modelos por varios datasets
    Devuelve los resultados y una summary table
    """

    results = []

    for dataset_name, data in datasets.items():

        X_train = data["X_train"]
        X_test = data["X_test"]

        for model_name, model in models.items():

            model_clone = clone(model)

            result = evaluate_model(
                model=model_clone,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                model_name=model_name,
                dataset_name=dataset_name
            )

            results.append(result)

    summary_df = summarize_results(results)

    return results, summary_df



#### Clasificador de Features

In [ ]:
def classify_feature_type(feature):
    if feature.startswith("diff_"):
        return "diff"
    elif feature.startswith("relative_diff_"):
        return "relative_diff"
    elif feature.startswith("sum_"):
        return "sum"
    elif feature.startswith("home_"):
        return "home_raw"
    elif feature.startswith("away_"):
        return "away_raw"
    else:
        return "other"
    
    

#### Contador de Features

In [105]:
def summarize_feature_types(feature_importance):
    """
    Summarizes feature types in the importance ranking.
    """

    summary = (
        feature_importance["feature_type"]
        .value_counts()
        .rename_axis("feature_type")
        .reset_index(name="count")
    )

    summary["percentage"] = (
        summary["count"] / summary["count"].sum()
    ).round(3)

    return summary



### Evaluación de Modelos

#### Instacio modelos

In [94]:
models = {
    "Dummy": DummyClassifier(
        strategy="most_frequent",
        random_state=42
    ),

    "Logistic Regression": LogisticRegression(
        random_state=42,
        max_iter=1000
    ),

    "Logistic Regression Scaled": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                random_state=42,
                max_iter=1000
            ))
        ]
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42
    )
}

#### Instancio Datasets

In [95]:
datasets = {
    "V01": {
        "X_train": X_train_v01,
        "X_test": X_test_v01
    },

    "V02": {
        "X_train": X_train_v02,
        "X_test": X_test_v02
    }
}

#### Resultados

In [96]:
results, summary_df = run_model_comparison(
    models=models,
    datasets=datasets,
    y_train=y_train,
    y_test=y_test
)


c:\Users\Guido\Documents\Gui Archives\Coding\Mundial 2026\world-cup-predictor\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Guido\Documents\Gui Archives\Coding\Mundial 2026\world-cup-predictor\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You mi

In [97]:
summary_df

,dataset,model_name,accuracy
0,V02,Random Forest,0.484375
1,V01,Random Forest,0.468750
2,V02,Dummy,0.453125
3,V01,Dummy,0.453125
4,V02,Logistic Regression,0.359375
5,V01,Logistic Regression,0.359375
6,V02,Logistic Regression Scaled,0.359375
7,V01,Logistic Regression Scaled,0.234375


El dummy da altísimo. ¿Hay una clase dominante?

In [98]:
y_train.value_counts(normalize=True)

target
 1    0.406250
-1    0.390625
 0    0.203125
Name: proportion, dtype: float64

In [99]:
display_results(results)

Dummy - V01
Accuracy: 0.453

Confusion Matrix:


,Pred -1,Pred 0,Pred 1
Real -1,0,0,20
Real 0,0,0,15
Real 1,0,0,29



Classification Report:


,precision,recall,f1-score,support
-1,0.000000,0.000000,0.000000,20.000000
0,0.000000,0.000000,0.000000,15.000000
1,0.453125,1.000000,0.623656,29.000000
accuracy,0.453125,0.453125,0.453125,0.453125
macro avg,0.151042,0.333333,0.207885,64.000000
weighted avg,0.205322,0.453125,0.282594,64.000000




Logistic Regression - V01
Accuracy: 0.359

Confusion Matrix:


...

,precision,recall,f1-score,support
-1,0.393939,0.650000,0.490566,20.000000
0,0.190476,0.266667,0.222222,15.000000
1,0.600000,0.206897,0.307692,29.000000
accuracy,0.359375,0.359375,0.359375,0.359375
macro avg,0.394805,0.374521,0.340160,64.000000
weighted avg,0.439624,0.359375,0.344808,64.000000




Random Forest - V02
Accuracy: 0.484

Confusion Matrix:


,Pred -1,Pred 0,Pred 1
Real -1,16,1,3
Real 0,10,0,5
Real 1,14,0,15



Classification Report:


,precision,recall,f1-score,support
-1,0.400000,0.800000,0.533333,20.000000
0,0.000000,0.000000,0.000000,15.000000
1,0.652174,0.517241,0.576923,29.000000
accuracy,0.484375,0.484375,0.484375,0.484375
macro avg,0.350725,0.439080,0.370085,64.000000
weighted avg,0.420516,0.484375,0.428085,64.000000


In [100]:
rf_v01_result = [
    result for result in results
    if result["model_name"] == "Random Forest" and result["dataset"] == "V01"
][0]

rf_v02_result = [
    result for result in results
    if result["model_name"] == "Random Forest" and result["dataset"] == "V02"
][0]

In [101]:
print(len(rf_v01_result["model"].feature_importances_))
print(len(X_train_v01.columns))

print(len(rf_v02_result["model"].feature_importances_))
print(len(X_train_v02.columns))

50
50
125
125


In [102]:
rf_v01_importance = get_feature_importance(
    rf_v01_result,
    X_train_v01.columns,
    top_n=20
)

rf_v02_importance = get_feature_importance(
    rf_v02_result,
    X_train_v02.columns,
    top_n=20
)

print("Random Forest V01 - Top Features")
display(rf_v01_importance)

print("Random Forest V02 - Top Features")
display(rf_v02_importance)

Random Forest V01 - Top Features


,feature,importance
0,home_Duel,0.040485
1,home_Ball Receipt*,0.038642
2,home_Ball Recovery,0.037328
3,away_Ball Recovery,0.034374
4,home_Pass,0.032697
5,home_Dispossessed,0.031457
6,home_Carry,0.030590
7,away_50/50,0.030193
8,home_Dribble,0.029797
9,away_Clearance,0.028366


Random Forest V02 - Top Features


,feature,importance
0,diff_Dispossessed,0.023942
1,home_Ball Recovery,0.021297
2,relative_diff_Dispossessed,0.020239
3,home_Duel,0.020211
4,diff_Ball Recovery,0.019539
5,diff_Clearance,0.019465
6,diff_Dribble,0.018807
7,diff_Duel,0.018540
8,relative_diff_Pressure,0.018286
9,sum_Error,0.018225


In [104]:
rf_v02_importance["feature_type"] = rf_v02_importance["feature"].apply(classify_feature_type)

rf_v02_importance

,feature,importance,feature_type
0,diff_Dispossessed,0.023942,diff
1,home_Ball Recovery,0.021297,home_raw
2,relative_diff_Dispossessed,0.020239,relative_diff
3,home_Duel,0.020211,home_raw
4,diff_Ball Recovery,0.019539,diff
5,diff_Clearance,0.019465,diff
6,diff_Dribble,0.018807,diff
7,diff_Duel,0.018540,diff
8,relative_diff_Pressure,0.018286,relative_diff
9,sum_Error,0.018225,sum


In [106]:
summarize_feature_types(rf_v02_importance)

,feature_type,count,percentage
0,diff,7,0.35
1,home_raw,6,0.30
2,relative_diff,5,0.25
3,sum,2,0.10


# Conclusión (Español)



## Objetivo

El objetivo de esta notebook fue evaluar si las variables generadas en `match_vector_v02` mejoraban el rendimiento predictivo respecto del dataset original `match_vector_v01`.

Las nuevas familias de variables evaluadas fueron:

- Diferencias absolutas (`diff_*`)
- Diferencias relativas (`relative_diff_*`)
- Variables de volumen (`sum_*`)

Se compararon tres modelos de clasificación:

- Dummy Classifier
- Logistic Regression
- Random Forest

El entrenamiento se realizó utilizando los datos del Mundial 2018, mientras que la evaluación se efectuó sobre el Mundial 2022.

---

## Resultados

La comparación mostró que el feature engineering incorporado en V02 produjo una mejora medible en los modelos basados en árboles.

| Modelo | Accuracy V01 | Accuracy V02 |
|---------|-------------:|-------------:|
| Dummy | 0.453 | 0.453 |
| Logistic Regression | 0.359 | 0.359 |
| Logistic Regression (Scaled) | 0.234 | 0.359 |
| Random Forest | **0.469** | **0.484** |

Si bien la mejora en accuracy fue moderada, Random Forest superó de forma consistente tanto al Dummy Baseline como a Logistic Regression.

---

## Análisis de Importancia de Variables

El análisis de importancia de variables mostró que las nuevas features pasaron a ser las principales fuentes de información utilizadas por el modelo.

Dentro de las 20 variables más importantes del Random Forest:

- 7 correspondieron a variables `diff_*`
- 5 correspondieron a variables `relative_diff_*`
- 2 correspondieron a variables `sum_*`
- 6 fueron variables originales (`home_*`)

Además, prácticamente no aparecieron variables originales `away_*` entre las más relevantes, lo que sugiere que las representaciones basadas en diferencias lograron resumir de forma más eficiente la relación entre ambos equipos.

Esto indica que las variables creadas no constituyen únicamente un aumento en la cantidad de features, sino nuevas representaciones que el modelo utiliza activamente para realizar sus predicciones.

---

## Conclusión

El feature engineering implementado en la Notebook 06 puede considerarse exitoso.

Aunque la mejora en accuracy fue relativamente moderada, el modelo Random Forest demostró que las nuevas variables aportan información predictiva relevante.

Por lo tanto, `match_vector_v02` será adoptado como el nuevo dataset base para las siguientes etapas del proyecto.

Las próximas versiones se enfocarán en:

- analizar estrategias de selección de variables;
- evaluar modelos más avanzados (por ejemplo, XGBoost);
- incorporar estadísticas futbolísticas de mayor riqueza para continuar mejorando la capacidad predictiva del modelo.

# Conclusion (English)

## Objective

The objective of this notebook was to evaluate whether the engineered features introduced in `match_vector_v02` improved the predictive performance compared to the original `match_vector_v01`.

The evaluated feature families were:

- Absolute differences (`diff_*`)
- Relative differences (`relative_diff_*`)
- Total volume features (`sum_*`)

Three classification models were compared:

- Dummy Classifier
- Logistic Regression
- Random Forest

Training was performed using the 2018 FIFA World Cup, while testing was conducted on the 2022 FIFA World Cup.

---

## Results

The comparison showed that the feature engineering introduced in V02 produced a measurable improvement for tree-based models.

| Model | V01 Accuracy | V02 Accuracy |
|--------|-------------:|-------------:|
| Dummy | 0.453 | 0.453 |
| Logistic Regression | 0.359 | 0.359 |
| Logistic Regression (Scaled) | 0.234 | 0.359 |
| Random Forest | **0.469** | **0.484** |

Although the improvement in accuracy was modest, Random Forest consistently outperformed both the Dummy baseline and Logistic Regression.

---

## Feature Importance Analysis

Feature importance analysis revealed that the newly engineered variables became the dominant predictors.

Among the Top 20 most important features of the Random Forest model:

- 7 were `diff_*` features
- 5 were `relative_diff_*` features
- 2 were `sum_*` features
- 6 were original (`home_*`) features

Notably, very few original `away_*` variables remained among the most important predictors, suggesting that difference-based representations successfully summarized the relationship between both teams.

This indicates that the engineered features were not merely additional variables, but informative representations actively used by the model.

---

## Conclusion

The feature engineering implemented in Notebook 06 is considered successful.

Although the overall accuracy improvement was relatively small, the Random Forest model demonstrated that the engineered variables contributed meaningful predictive information.

Therefore, `match_vector_v02` will be adopted as the baseline dataset for subsequent stages of the project.

Future work will focus on:

- analyzing feature selection strategies;
- evaluating more powerful models (e.g. XGBoost);
- expanding the feature set with richer football-related statistics.